In [ ]:
%pip install pandas

In [ ]:
import os
import pandas as pd
import utils.age_limit_processing as alp

year = 2015

items_path = f"../data/processed/MAL/items_{year}_enriched.tsv"

risk_items_save_path = f"../data/processed/MAL/risk_items_{year}.tsv"

cover_art_dir = f"../data/processed/MAL/cover_art/"


In [ ]:
if os.path.exists(risk_items_save_path):
    items = pd.read_csv(risk_items_save_path, sep="\t")
else:
    items = pd.read_csv(items_path, sep="\t")

In [ ]:
from risk_detection import ModerationClassifier

moderator = ModerationClassifier()



In [ ]:
from tqdm.auto import tqdm

for index, row in tqdm(items.iterrows(), total=len(items), desc="Processing items"):
    text = row['synopsis']
    
    if not isinstance(text, str) or not text.strip():
        # print(f"Image for item {row['item_id']} not found. Skipping.")
        items.at[index, "text_sexual"] = 0
        items.at[index, "text_violence"] = 0
        continue
    text_sexual, text_violence = moderator.predict_S_V(text)
    
    items.at[index, "text_sexual"] = text_sexual
    items.at[index, "text_violence"] = text_violence
    
    # print(f"Processed item {row['item_id']} with sexual score: {text_sexual}")
    # print(f"Processed item {row['item_id']} with violence score: {text_violence}")

In [ ]:
items['age_rating'] = items['age_rating_raw']
items['age_rating'].fillna('Not Rated', inplace=True)

In [ ]:
items.columns

In [ ]:
items = items[['item_id', 'title', 'title_english', 'age_rating_raw', 'age_rating', 'genres',
       'synopsis', 'text_sexual', 'text_violence']]

In [ ]:
items.to_csv(risk_items_save_path, sep="\t", index=False)